# Improvements over the original baseline

This notebook implements a LightGBM model. The main improvements compared to the original notebook are:

- **Session-safe train/test split**: all requests from one session stay in either training or testing, reducing data leakage and producing a more realistic evaluation.
- **Preprocessing pipeline**: OneHotEncoder is fitted only on training data and safely handles unseen categories. pd.get_dummies() was applied before the split in the original notebook.
- **LightGBM instead of KNN**: LightGBM is generally more suitable for nonlinear relationships, mixed feature importance, and larger datasets. KNN prediction becomes slower as training data grows.
- **More meaningful evaluation**: macro F1 evaluates every class equally instead of relying only on accuracy, which can hide poor performance on rare classes.
- **Reproducibility**: the split and model use a fixed random_state.
- **Explicit class-imbalance analysis**: you identified that balanced class weights are harmful because some classes contain almost no data.
- **Safer handling of missing values**: missingness is preserved instead of automatically converting every missing value to an empty string.

# Read and explore data

In [1]:
import pandas as pd

df = pd.read_csv("../data/clickdata.csv")

In [2]:
print(df.shape)
print(df["ua_agent_class"].value_counts())

(59782, 8)
ua_agent_class
Browser              35509
Robot                16026
Robot Mobile          5115
Browser Webview       1800
Hacker                1177
Special                144
Mobile App               9
Cloud Application        2
Name: count, dtype: int64


In [3]:
# inspect label: evaluate class imbalance to determine how to evaluate the model
print(df["ua_agent_class"].value_counts(normalize=True, dropna=False).mul(100))

ua_agent_class
Browser              59.397478
Robot                26.807400
Robot Mobile          8.556087
Browser Webview       3.010940
Hacker                1.968820
Special               0.240875
Mobile App            0.015055
Cloud Application     0.003345
Name: proportion, dtype: float64


### Missing values

I preserve missing categorical values rather than replacing them with empty strings. OneHotEncoder treats missing values as a separate category, allowing the model to learn whether missingness is informative.

In [4]:
#inspect missing values. But dont blindly convert missing values to empty strings. 

missing_summary = pd.DataFrame({"missing": df.isna().sum(),"unknown": (df == "Unknown").sum(),"empty": (df == "").sum()})

display(missing_summary[(missing_summary["missing"] > 0) | (missing_summary["unknown"] > 0) | (missing_summary["empty"] > 0)])

,missing,unknown,empty
country_by_ip_address,248,0,0
region_by_ip_address,9896,0,0
referrer_without_parameters,44778,0,0


# Define the problem

The original data contains several user-agent classes.

Similar to original notebook, I normalize closely related classes:

- `Browser Webview` → `Browser`
- `Robot Mobile` → `Robot`

The resulting target is treated as a multiclass classification problem.

I will retain the remaining classes rather than collapsing everything into a binary bot/non-bot target, because the assignment explicitly asks to distinguish different types of non-human traffic. So we have 6 labels:

- Browser
- Robot
- Hacker
- Special
- Mobile App
- Cloud Application

In [5]:
# Merge all different Human types.  
# Merge all different 'non hunam' types
df["target"] = df["ua_agent_class"].replace({"Browser Webview": "Browser","Robot Mobile": "Robot"})

# inspect target
print(df["target"].value_counts(normalize=True, dropna=False).mul(100))

target
Browser              62.408417
Robot                35.363487
Hacker                1.968820
Special               0.240875
Mobile App            0.015055
Cloud Application     0.003345
Name: proportion, dtype: float64


In [6]:
df['target'].nunique()

6

# Train LightGBM: version1

In [7]:
# consistent with the original notebook
feature_columns = ["country_by_ip_address", "region_by_ip_address", "visitor_recognition_type",]

X = df[feature_columns]
y = df["target"]

### train/test split

A random row split can place requests from the same session in both training and test sets. As requests within a session are highly related, this causes data leakage and produces overly optimistic evaluation results.

I therefore split by session_id, keeping every session entirely in either training or test data while approximately preserving class distributions.

In [8]:
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

train_idx, test_idx = next(splitter.split(X, y, groups=df["session_id"]))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

C:\Users\P313204\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_split.py:1036: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


Because some classes contain very few observations, the grouped split cannot include every class reliably.

### preprocess categorical features

I used OneHotEncoder instead of pd.get_dummies() because it can be fitted only on the training data and then applied consistently to the test data. It also handles categories that appear only in the test set using handle_unknown="ignore", preventing preprocessing leakage and column mismatches.
categorical_transformer = OneHotEncoder(handle_unknown="ignore")


LightGBM supports categorical features natively, so one-hot encoding is not strictly required. I use it here to create a complete preprocessing pipeline that safely handles unseen categories and can be reused by the API.

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(transformers=[("categorical", categorical_transformer, feature_columns)])

In [10]:
# Define the LightGBM model
from lightgbm import LGBMClassifier

model = LGBMClassifier(
    objective="multiclass",
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

In [11]:
# Create and train the pipeline
from sklearn.pipeline import Pipeline

baseline_pipeline = Pipeline([("preprocessing", preprocessor), ("classifier", model)])

baseline_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](6,)","['Browser','Cloud Application','Hacker','Mobile App','Robot','Special']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['country_by_ip_address','region_by_ip_address','visitor_recognition_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (d

In [12]:
# Evaluate the model

from sklearn.metrics import confusion_matrix

y_pred = baseline_pipeline.predict(X_test)
confusion_matrix(y_test, y_pred)

array([[7401,    0,    0,   60,    0],
       [  79,    0,    0,  156,    0],
       [   1,    0,    0,    0,    0],
       [  27,    0,    0, 4202,    0],
       [  14,    0,    0,    9,    5]])

In [13]:
from sklearn.metrics import f1_score

macro_f1 = f1_score(y_test, y_pred, average="macro")

print("Macro F1:", macro_f1)

Macro F1: 0.4523674382265136


In [14]:
accuracy = baseline_pipeline.score(X_test, y_test)

print("Accuracy:", accuracy)

Accuracy: 0.9710557135686799


## Further Improvements

The first version only uses country, region, and visitor recognition type. This provides a simple baseline, but these features contain limited information about user behaviour.

A stronger model could include:

- Temporal features: time since the start of the session and time since the previous request. Bots may generate requests at more regular or faster intervals than humans.
- Session features: number of requests, number of unique URLs, session duration, and request frequency. These features can capture repetitive or unusually intensive activity.
- URL features: URL length, path depth, number of digits, and page type. Bots and humans may access different types of pages.
- Referrer features: whether a referrer exists, whether it is internal, and whether it matches the requested URL. These features provide information about navigation behaviour.

These features should be created carefully to avoid using future requests when making a real-time prediction. Model tuning and alternative approaches to class imbalance could also be explored. However, the extremely rare classes should first be discussed with the data owner because there may not be enough examples to train and evaluate them reliably.

Some target classes contain too few examples to train and evaluate reliably. For example, Mobile App and Cloud Application contain only a handful of observations. A future version could combine rare classes into an Other category. This would reduce instability and give the model more examples for the combined class. However, classes should only be merged if they have a meaningful business relationship. The final grouping should therefore be agreed with the data owner rather than based only on class frequency.

# binary LightGBM model for the API

We need to train a separate binary LightGBM model for the API. The binary target is:

- **NHT**
- **not-NHT** or **HT**
  
For the API model, I map Browser, Browser Webview, and provisionally Mobile App to human traffic (non-nht/ht). All remaining classes are mapped to NHT. The treatment of Mobile App and Special should be confirmed with the data owner.

In [15]:
# create the binary target

human_classes = ["Browser", "Browser Webview", "Mobile App",]

df["is_nht"] = (~df["ua_agent_class"].isin(human_classes)).astype(int)

In [36]:
df["is_nht"].value_counts(normalize=True, dropna=False).mul(100)

is_nht
0    62.423472
1    37.576528
Name: proportion, dtype: float64

In [16]:
# split the data using the new target

X = df[feature_columns]
y = df["is_nht"]

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

train_idx, test_idx = next(splitter.split(X, y, groups=df["session_id"]))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

In [17]:
# train LightGBM with a binary objective

binary_model = LGBMClassifier(
    objective="binary",
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

In [18]:
# keep the preprocess and the new model in a same pipeline

binary_pipeline = Pipeline([("preprocessing", preprocessor), ("classifier", binary_model)])
binary_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](3,)","['country_by_ip_address','region_by_ip_address','visitor_recognition_type']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,3
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, al

In [19]:
y_pred = binary_pipeline.predict(X_test)

macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
accuracy = binary_pipeline.score(X_test, y_test)

print(f"Accuracy: {accuracy:.3f}")
print(f"Macro F1: {macro_f1:.3f}")

Accuracy: 0.982
Macro F1: 0.981


In [33]:
# predict an individual data record
row_position = 200

y_pred = binary_pipeline.predict(X_test.iloc[[row_position]])[0]

y_real = y_test.iloc[row_position]

print("Predicted:", y_pred)
print("Actual:", y_real)

Predicted: 0
Actual: 0


In [27]:
# save the pipeline
import joblib

joblib.dump(binary_pipeline, "../models/bot_detection_pipeline.joblib")

['../models/bot_detection_pipeline.joblib']

In [28]:
# verify the pipeline can be loaded
loaded_pipeline = joblib.load("../models/bot_detection_pipeline.joblib")

loaded_pipeline.predict(X_test.head())

array([1, 1, 0, 0, 1])